# Asistente de Histología (ETL & Chat)
Este notebook te permite inicializar el entorno, clonar tu proyecto estructural de Github, instalar dependencias de sistema y de Python, ingestar los PDFs y correr tu CLI interactivo de inferencia RAG.
Asegúrate de configurar tu sesión de Colab con un acelerador de hardware **GPU (ej. T4)**.

In [1]:
import os

# 1. Instalar pre-requisitos de sistema operativo (vitales para OCR e Imágenes) y UV para rapidez
!sudo apt-get update -qq && sudo apt-get install -y poppler-utils tesseract-ocr -qq
!pip install uv -q

# 2. Clonar el repositorio usando el token de Github en Secretos de Colab
from google.colab import userdata
github_token = userdata.get('GITHUB_TOKEN')
github_user = "tompivel"   # <-- Reemplaza por tu usuario/repo si hace falta
repo_name = "histo-test"
repo_url = f"https://{github_token}@github.com/{github_user}/{repo_name}.git"

if not os.path.exists(repo_name):
    !git clone -b refactoring --single-branch {repo_url} -q

%cd {repo_name}

# 3. Instalar dependencias puras mediante UV basándose en el pyproject.toml
!uv pip install --system -r pyproject.toml

import nest_asyncio
nest_asyncio.apply()

# 4. RESURRECT 'imp' TO FIX AUTORELOAD IN PYTHON 3.12
!uv pip install --system zombie-imp

# 5. This tells Colab to watch your .py files for changes and
# automatically reload them before executing any new code.
%load_ext autoreload
%load_ext autoreload
%autoreload 2

print("✅ Entorno preparado exitosamente.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package poppler-utils.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━

### Paso 1: Ejecutar Ingesta (Data ETL)
Analiza PDFs, aplica OCR, calcula vectores Texto, UNI y PLIP y rellena Neo4J. *(Solo correr cuando debas actualizar el Graph Database)*.

In [2]:
from core.ingestion import PipelineIngestion
from utils.config import userdata
import os

async def run_etl():
    print("Levantando entorno de Data Pipeline...")
    pipeline = PipelineIngestion(
        neo4j_uri=userdata.get('NEO4J_URI'),
        neo4j_user=userdata.get('NEO4J_USERNAME'),
        neo4j_pass=userdata.get('NEO4J_PASSWORD'),
        device='cuda'
    )
    await pipeline.inicializar()

    # Generamos la carpeta de escucha si no existe:
    directorio_pdfs = os.path.join(os.getcwd(), 'pdf')
    os.makedirs(directorio_pdfs, exist_ok=True)
    print(f"Coloca tus manuales PDF en: {directorio_pdfs} antes de continuar.\n")

    # force=True si quieres sobrescribir datos
    await pipeline.ejecutar(directorio_pdfs=directorio_pdfs, forzar=False)
    await pipeline.cerrar()

# -----------------------------------------------------
#  Si deseas poblar Neo4j, descomenta la siguiente línea:
# -----------------------------------------------------
await run_etl()

✅ Logueado en Hugging Face
✅ LangSmith — Proyecto: rag_histologia_neo4j_v4
Levantando entorno de Data Pipeline...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Conectando a la Base de Datos Neo4j y cargando modelos locales...
✅ Neo4j conectado: neo4j+s://30af270f.databases.neo4j.io
🏗️ Creando esquema Neo4j (v4.2 UNI + PLIP)...
✅ Esquema Neo4j listo (3 índices)
🔄 Cargando UNI (MahmoodLab)...
✅ UNI cargado
🔄 Cargando PLIP (vinid/plip)...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: vinid/plip
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


✅ PLIP cargado
Coloca tus manuales PDF en: /content/histo-test/pdf antes de continuar.

✅ Base de datos Neo4j ya poblada. Saltando indexación. (Usa forzar=True)


### Paso 2: Chat Interactivo (RAG)
Instancia la red nodal en LangGraph. Podrás escribir textos directo desde el Colab, o enviarle path a imágenes subidas al entorno `/content/`.

In [2]:
from core.cli import modo_interactivo

# Inicia el bucle eterno de consulta conversacional
await modo_interactivo()

✅ Logueado en Hugging Face
✅ LangSmith — Proyecto: rag_histologia_neo4j_v4

🔬 ASISTENTE DE HISTOLOGÍA - RAG MULTIMODAL GROQ (v4.3) 🔬
Usando Llama-4-17B, LangGraph, UNI + PLIP, Neo4j, Qdrant
------------------------------------------------------------
Comandos:
  'salir'    -> Terminar la sesión
  'imagen <ruta>' -> Subir una imagen junto con tu próxima pregunta (Ej: imagen /tmp/foto.jpg)
    
🖥️ Inicializando asistente en: CUDA
✅ Neo4j conectado: neo4j+s://30af270f.databases.neo4j.io
🧠 Inicializando LLM Central (Groq: Llama-4-17B)...
🧠 Inicializando Embeddings de Texto Local...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

✅ PLIP cargado
✅ Modelos base listos.
🏗️ Creando esquema Neo4j (v4.2 UNI + PLIP)...
✅ Esquema Neo4j listo (3 índices)
✅ LangGraph compilado (v4.2)

👤 Tu consulta: imagen /content/histo-test/imagenes_chat/upload_77c47a5c.jpg
✅ Imagen 'upload_77c47a5c.jpg' cargada en memoria temporal.

[📷 Imagen cargada y lista para la siguiente pregunta]

👤 Tu consulta: ¿podrías explicarme las características del tejido de la imagen?
⏳ Analizando y procesando (LangGraph en curso)...
   🔍 [Nodo] Verificando dominio...
   🔄 Precalculando embeddings de anclas semánticas...
   📐 Similitud semántica con dominio histológico: 0.8108
   👁️ [Nodo] Procesando imagen del usuario...
   🧠 [Nodo] Recuperando contexto de memoria...
   🔎 [Nodo] Ejecutando búsqueda híbrida multispace...


   📊 Híbrida: Txt=8 | UNI=0 | PLIP=8 | Ent=8 | Vec=13 -> 23
   ✍️ [Nodo] Generando respuesta final (Rol: Juez Escéptico)...
   🧠 Interacción guardada en memoria semántica (ID: 0)

🧠 ASISTENTE:
Análisis de la Imagen Histológica

La imagen histológica subida por el usuario parece corresponder a un tejido conjuntivo o conectivo, posiblemente de tipo fibroso. A continuación, te proporcionaré detalles sobre las características celulares, tinciones probables, estructuras visibles y posibles diagnósticos/tejidos.

**Características Celulares:**

*   Las células observadas en la imagen son principalmente fibroblastos, que son células típicas del tejido conjuntivo. Estos fibroblastos tienen núcleos alargados y citoplasma poco visible, lo que es característico en tejidos conjuntivos fibrosos.
*   No se observan células inflamatorias significativas, como linfocitos o neutrófilos, lo que sugiere la ausencia de un proceso inflamatorio agudo.
*   No hay evidencia de malignidad celular, como pleomorf

   📊 Híbrida: Txt=8 | UNI=0 | PLIP=8 | Ent=0 | Vec=14 -> 30
   ✍️ [Nodo] Generando respuesta final (Rol: Juez Escéptico)...
   🧠 Interacción guardada en memoria semántica (ID: 1)

🧠 ASISTENTE:
Lo siento, pero no puedo proporcionar una descripción histológica de la imagen que has proporcionado porque la imagen no muestra una preparación histológica. En su lugar, parece ser una fotografía de una persona.

Si deseas describir una imagen histológica, necesitaría ver una imagen de un corte histológico teñido, como uno teñido con Hematoxilina y Eosina (H&E), que es una de las tinciones más comunes en histología.

Si puedes proporcionar la imagen de un corte histológico o más detalles sobre lo que estás buscando, estaré encantado de ayudarte con la descripción.

Referencias:
- Gartner, L. P., Nava, A. S., Isabel, G. P. M., Ángel, H. E. M., & Roig, G. F. (2018). Histología: Atlas en color y texto. (7a. Ed.). Wolters Kluwer.
- I. F. van der G. T. (2017). Histología y Biología Celular. (3a. Ed.)

CancelledError: 